In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [2]:
warnings.filterwarnings('ignore')
df = pd.read_excel('Financial_Loan.xlsx')

## Understanding & Data Preperation

In [3]:
df.head(3)

,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,2021-02-11,2021-09-13,2021-04-13,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,2021-01-01,2021-12-14,2021-01-15,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,2021-01-05,2021-12-12,2021-01-09,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522


In [4]:
df.columns

Index(['id', 'address_state', 'application_type', 'emp_length', 'emp_title',
       'grade', 'home_ownership', 'issue_date', 'last_credit_pull_date',
       'last_payment_date', 'loan_status', 'next_payment_date', 'member_id',
       'purpose', 'sub_grade', 'term', 'verification_status', 'annual_income',
       'dti', 'installment', 'int_rate', 'loan_amount', 'total_acc',
       'total_payment'],
      dtype='object')

In [5]:
print(f'Number of Rows: {df.shape[0]:,}')
print(f'Number of Columns: {df.shape[1]:,}')

Number of Rows: 38,576
Number of Columns: 24


In [6]:
print(f'Duplicate Values are: {df.duplicated().sum()}')

Duplicate Values are: 0


In [7]:
print(f'Number of Null Values: {df.isnull().sum().sum():,}')

Number of Null Values: 1,438


In [8]:
df.isnull().sum()

id                          0
address_state               0
application_type            0
emp_length                  0
emp_title                1438
grade                       0
home_ownership              0
issue_date                  0
last_credit_pull_date       0
last_payment_date           0
loan_status                 0
next_payment_date           0
member_id                   0
purpose                     0
sub_grade                   0
term                        0
verification_status         0
annual_income               0
dti                         0
installment                 0
int_rate                    0
loan_amount                 0
total_acc                   0
total_payment               0
dtype: int64

In [9]:
# Fill Missing Values
df['emp_title'] = df['emp_title'].fillna('Unknown')

In [10]:
# Convert 'term' from string format ("36 months") to integer
df['term'] = df['term'].str.strip().str.replace('months', '').astype(int)

## Data Analysis

### Overall Portfolio Performance

    Total loans issued, total amount lent, average loan size, average interest rate

In [11]:
print(f'Total Loans Issued: {df.shape[0]:,}')
print(f'Total Amount Lent: ${df['loan_amount'].sum():,}')
print(f'Average Loan Size: ${df['loan_amount'].mean():,.2f}')
print(f'Average Interest Rate: {df['int_rate'].mean():,.2%}')
print(f'Total Payment Received: ${df['total_payment'].sum():,}')

Total Loans Issued: 38,576
Total Amount Lent: $435,757,075
Average Loan Size: $11,296.07
Average Interest Rate: 12.05%
Total Payment Received: $473,070,933


    Distribution of loan_status (Fully Paid, Charged Off, Late, etc.)

In [12]:
round(df['loan_status'].value_counts(normalize=True) * 100, 2)

loan_status
Fully Paid     83.33
Charged Off    13.82
Current         2.85
Name: proportion, dtype: float64

In [13]:
fig = go.Figure(go.Pie(
    labels=df['loan_status'].value_counts().index,
    values=df['loan_status'].value_counts().values,

    hole=0.55,
    pull=[0.04, 0.04, 0.04],
    textinfo='percent',
    textfont=dict(size=14, color='white'),

    hovertemplate="<b>%{label}</b><br>Profit: $%{value:,.0f}<br>%{percent}<extra></extra>",
    marker=dict(colors=['#0A2540', '#1E88E5', 'green', '#FB8C00', '#43A047'],
    line=dict(color='white', width=2))
    ))
fig.update_layout(
    title=dict(
        text="<b>Loan Status Distribution</b>",
        x=0.5,
        font=dict(size=22)),
   annotations=[dict(
        text="<b>Loan<br>Status</b>",
        x=0.5,
        y=0.5,
        font=dict(size=18),
        showarrow=False)],
    showlegend=True,
    legend=dict(
        orientation="v",
        x=0.74,
        y=1.15,
        font=dict(size=15)))
fig.show()

    Default / Charged Off rate overall and by grade

In [14]:
bad_loans = df[df['loan_status'].isin(['Charged Off', 'Default'])]
default_rate = len(bad_loans) / len(df)

print(f'Overall Default Rate: {default_rate * 100:.2f} %')

Overall Default Rate: 13.82 %


In [15]:
default_by_grade = df.groupby('grade')['loan_status'].apply(lambda x: x.isin(['Charged Off', 'Default']).mean() * 100).reset_index()

default_by_grade.columns = ['Grade', 'Default Rate']
default_by_grade

,Grade,Default Rate
0,A,5.697182
1,B,11.504197
2,C,16.017206
3,D,20.686993
4,E,24.802584
5,F,30.252918
6,G,31.309904


In [16]:
fig = px.bar(
    default_by_grade,
    x='Grade',
    y='Default Rate',
    text='Default Rate',
    color='Grade',
    title='<b>Default Rate by Loan Grade</b>')

fig.update_traces(textposition='outside',
                  texttemplate='<b>%{text:.2f}%</b>', 
                  hovertemplate='<b>Grade:</b> %{x}<br><b>Rate:</b> %{y:,.2f}%<br><extra></extra>',
                  hoverlabel=dict(font_color='white'))
fig.update_layout(
    title=dict(text='<b>Default Rate by Grade</b>',x=0.5,font=dict(size=22)),
    xaxis_title=dict(text='<b>Loan Grade</b>',font=dict(size=18)),
    yaxis=dict(
        title=dict(text='<b>Default Rate (%)</b>', font=dict(size=16)),tickfont=dict(size=14),ticksuffix='%',showgrid=True),
    template='simple_white',
    legend=dict(
        title='<b>Grade</b>',
        x=0.99,
        y=1.2,
        orientation='v',
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='black',
        borderwidth=1),
    bargap=0.45)
fig.show()

### Customer & Risk Segmentation

    Default rate by grade and sub_grade 

In [17]:
grade_subgrade_rate = df.groupby(['grade', 'sub_grade'])['loan_status'].apply(lambda x: (x == 'Charged Off').mean() * 100).reset_index(name='Default Rate')
grade_subgrade_rate = grade_subgrade_rate.rename(columns={'grade': 'Grade','sub_grade': 'Sub Grade'})

grade_subgrade_rate

,Grade,Sub Grade,Default Rate
0,A,A1,2.281369
1,A,A2,4.652778
2,A,A3,5.114943
3,A,A4,5.993578
4,A,A5,7.686511
5,B,B1,8.680754
6,B,B2,10.552764
7,B,B3,11.467890
8,B,B4,12.708758
9,B,B5,13.010590


In [18]:
import plotly.graph_objects as go

# Create hierarchical structure
labels = []
parents = []
values = []
colors = []

# Add Grade level
for grade in grade_subgrade_rate['Grade'].unique():
    grade_df = grade_subgrade_rate[grade_subgrade_rate['Grade'] == grade]

    labels.append(grade)
    parents.append("")
    values.append(grade_df['Default Rate'].sum())   # total of subgrades
    colors.append(grade_df['Default Rate'].mean())  # avg color

    # Add Sub-grade level
    for _, row in grade_df.iterrows():
        labels.append(row['Sub Grade'])
        parents.append(row['Grade'])
        values.append(row['Default Rate'])
        colors.append(row['Default Rate'])

In [19]:
fig = go.Figure(go.Sunburst(

    labels=labels,
    parents=parents,
    values=values,

    branchvalues='total',
    textinfo='label+percent entry',

    marker=dict(
        colors=colors,
        colorscale='twilight',
        colorbar=dict(title='<b>Default %</b>')),

    hovertemplate=
    '<b>Segment:</b> %{label}<br>' +
    '<b>Parent:</b> %{parent}<br>' +
    '<b>Default Rate:</b> %{value:.2f}%<br>' +
    '<b>Share:</b> %{percentEntry:.2%}<extra></extra>'
))
fig.update_layout(
    title=dict(
        text='<b>Default Risk by Grade & Sub-Grade</b>',x=0.5,font=dict(size=20)),
    margin=dict(t=90, l=10, r=10, b=50),
    template='plotly_white')
fig.show()

    Default rate by home_ownership and verification_status

In [20]:
home_ver_data = df.groupby(['home_ownership', 'verification_status'])['loan_status'] \
            .apply(lambda x: (x == 'Charged Off').mean() * 100).reset_index(name='Default Rate')
# Drop Null Values
home_ver_data = home_ver_data.drop(index=3).reset_index(drop=True)

In [21]:
home_ver_data

,home_ownership,verification_status,Default Rate
0,MORTGAGE,Not Verified,11.569886
1,MORTGAGE,Source Verified,12.238120
2,MORTGAGE,Verified,14.914463
3,OTHER,Not Verified,19.230769
4,OTHER,Source Verified,10.000000
5,OTHER,Verified,19.444444
6,OWN,Not Verified,12.191582
7,OWN,Source Verified,14.677641
8,OWN,Verified,16.689466
9,RENT,Not Verified,12.767519


In [22]:
color_map = {
    'Not Verified': '#0A2540',
    'Source Verified': 'chartreuse',
    'Verified': 'deepskyblue'
    }
fig = go.Figure()

verifications = home_ver_data['verification_status'].unique()
for v in verifications:
    temp = home_ver_data[home_ver_data['verification_status'] == v]
    
    fig.add_trace(go.Bar(
        x=temp['home_ownership'],
        y=temp['Default Rate'],
        name=v,

        text=temp['Default Rate'].round(2),
        texttemplate='<b>%{text}%</b>',
        textposition='outside',

        marker=dict(color=color_map.get(v)),
        hovertemplate="<b>%{x}</b><br>Default Rate: %{y:.2f}%<extra></extra>"
        ))
fig.update_layout(
    title=dict(text='<b>Default Rate by Home Ownership & Verification Status</b>', x=0.5, font=dict(size=20)),
    xaxis=dict(title=dict(text='<b>Home Ownership</b>', font=dict(size=18)),tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='<b>Default Rate (%)</b>', font=dict(size=18)),showgrid=True),

    barmode='group',
    template='simple_white',

    legend_title='<b>Verification Status</b>',
    legend=dict(
        x=0.85,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'),
    hovermode='x unified')
fig.show()

    Debt-to-Income Ratio (DTI) vs default rate — is high DTI really dangerous?

In [23]:
df['dti_bin'] = pd.cut(df['dti'],bins=[0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3],
    labels=['0-5%', '5-10%', '10-15%', '15-20%', '20-25%', '25-30%'])

dti_default = df.groupby('dti_bin')['loan_status'].apply(lambda x: (x == 'Charged Off').mean() * 100).reset_index(name='Default Rate')

dti_default

,dti_bin,Default Rate
0,0-5%,11.687500
1,5-10%,12.107280
2,10-15%,13.871335
3,15-20%,14.958762
4,20-25%,15.929337
5,25-30%,12.229102


In [24]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=dti_default['dti_bin'],
    y=dti_default['Default Rate'],

    text=dti_default['Default Rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    marker=dict(
        color=dti_default['Default Rate'],   
        colorscale='Reds'),
    hovertemplate='<b>DTI Level:</b> %{x}<br><b>Default Rate:</b> %{y:.2f}%<extra></extra>'))

avg_default = dti_default['Default Rate'].mean()

fig.add_hline(
    y=avg_default,
    line_dash="dash",
    line_color="black",

    annotation_text=f"<b>Avg: {avg_default:.2f}%<b>",
    annotation_position="top right")
    
fig.update_layout(
    title=dict(text='<b>Default Rate by DTI Level</b>', x=0.5, font=dict(size=22)),
    xaxis=dict(
        title=dict(text='<b>DTI Level</b>', font=dict(size=18)),tickfont=dict(size=14)),
    yaxis=dict(
        title=dict(text='<b>Default Rate (%)</b>', font=dict(size=18)),tickfont=dict(size=14),ticksuffix='%',showgrid=True),
    bargap=0.3,
    template='plotly_white')
fig.show()

High DTI is risky up to a point (20–25%), but in the highest DTI range, the risk does not continue increasing, likely due to stricter lending approval criteria.

    Annual income vs default rate 

In [25]:
df['income_bin'] = pd.cut(df['annual_income'],bins=[0, 30000, 60000, 100000, 200000, 1000000],
    labels=['<30K', '30K-60K', '60K-100K', '100K-200K', '200K+'])

income_default = df.groupby('income_bin')['loan_status'].apply(lambda x: (x == 'Charged Off').mean() * 100).reset_index(name='Default Rate')

income_default

,income_bin,Default Rate
0,<30K,17.325444
1,30K-60K,15.343236
2,60K-100K,12.164131
3,100K-200K,10.458679
4,200K+,10.283688


In [26]:
fig = go.Figure(go.Bar(
    x=income_default['income_bin'],
    y=income_default['Default Rate'],

    text=income_default['Default Rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    marker=dict(color=income_default['Default Rate'],colorscale='Reds'),

    hovertemplate='<b>Income:</b> %{x}<br><b>Default Rate:</b> %{y:.2f}%<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>Default Rate by Income Level</b>', x=0.5,font=dict(size=22)),
    xaxis=dict(title=dict(text='<b>Annual Income</b>',font=dict(size=18)),tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='<b>Default Rate (%)</b>',font=dict(size=18)),tickfont=dict(size=14),ticksuffix='%',showgrid=True),
    template='simple_white',
    bargap=0.45)
fig.show()

    Employment length (emp_length) vs default rate and average loan amount

In [27]:
def clean_emp_length(x):
    if x == '< 1 year':
        return 0.5
    elif x == '10+ years':
        return 10
    else:
        return float(x.split()[0])

df['emp_length_clean'] = df['emp_length'].apply(clean_emp_length)
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)

In [28]:
emp_analysis = df.groupby('emp_length_clean').agg(default_rate=('is_default', lambda x: x.mean() * 100),
                                                  avg_loan=('loan_amount', 'mean'),
                                                  total_loans=('id', 'count')).reset_index()
emp_analysis

,emp_length_clean,default_rate,avg_loan,total_loans
0,0.5,13.792350,9663.524590,4575
1,1.0,13.781356,10183.686900,3229
2,2.0,12.802373,10261.975126,4382
3,3.0,13.405088,10748.006360,4088
4,4.0,13.243874,10968.604142,3428
5,5.0,13.718301,11296.555148,3273
6,6.0,13.734291,11495.803411,2228
7,7.0,14.785553,11744.765801,1772
8,8.0,13.550136,11896.307588,1476
9,9.0,12.350598,12019.302789,1255


In [29]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=emp_analysis['emp_length_clean'],
    y=emp_analysis['default_rate'],
    
    name='<b>Default Rate (%)</b>',
    text=emp_analysis['default_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    customdata=emp_analysis[['total_loans']],
    hovertemplate=
    '<b>Employment Length:</b> %{x} years<br>' +
    '<b>Default Rate:</b> %{y:.2f}%<br>' +
    '<b>Total Loans:</b> %{customdata:,}<extra></extra>',
    
    marker=dict(color=emp_analysis['emp_length_clean'], colorscale = 'thermal'),
    yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=emp_analysis['emp_length_clean'],
    y=emp_analysis['avg_loan'],
    
    name='<b>Avg Loan Amount</b>',
    mode='lines+markers',
    text=emp_analysis['avg_loan'].round(0),
    hovertemplate='$%{y:,.0f}',
    
    line=dict(color='lime', width=4, ),
    marker=dict(size=8, color='#0A2540', line=dict(width=2, color='white')),
    fillcolor='rgba(30,136,229,0.15)',
    yaxis='y2'
))
fig.update_layout(
    title=dict(text='<b>Employment Length vs Default Risk & Loan Size</b>', x=0.5),
    xaxis=dict(title='<b>Employment Length (Years)</b>', range=[0,10.5],tickfont=dict(size=14)),
    yaxis=dict(
        title='<b>Default Rate (%)</b>',
        side='left',
        ticksuffix='%',
        tickfont=dict(size=14),
        showgrid=True),
    yaxis2=dict(
        title='<b>Avg Loan Amount</b>',
        overlaying='y',
        side='right',
        tickfont=dict(size=14)),
    template='simple_white',
    legend=dict(
        x=0.8,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'),
    bargap=0.05,
    hovermode='x unified')
fig.show()

    Loan purpose vs default rate and average int_rate (top 10 purposes)

In [30]:
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)

purpose_analysis = df.groupby('purpose').agg(default_rate=('is_default', lambda x: x.mean() * 100),
                                            avg_int_rate=('int_rate', 'mean'),
                                            total_loans=('id', 'count')).reset_index()
                                                                      
top_purpose = purpose_analysis.sort_values('total_loans', ascending=False).head(10)

In [31]:
top_purpose

,purpose,default_rate,avg_int_rate,total_loans
0,Debt consolidation,14.554738,0.125043,18214
2,credit card,10.164066,0.117307,4998
9,other,15.350418,0.118585,3824
4,home improvement,11.369958,0.114013,2876
6,major purchase,9.763033,0.108690,2110
11,small business,25.619369,0.130295,1776
1,car,10.354041,0.105921,1497
13,wedding,9.267241,0.118930,928
7,medical,14.992504,0.115709,667
8,moving,15.026834,0.115920,559


In [32]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_purpose['purpose'],
    y=top_purpose['default_rate'],
    
    name='<b>Default Rate (%)</b>',
    
    text=top_purpose['default_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    customdata=top_purpose[['avg_int_rate', 'total_loans']],
    hovertemplate=
    '<b>Purpose:</b> %{x}<br>' +
    '<b>Default Rate:</b> %{y:.2f}%<br>' +
    '<b>Total Loans:</b> %{customdata[1]:,}<extra></extra>',

    marker=dict(color=top_purpose['default_rate'],colorscale='ylorrd'),
    yaxis='y1'))
    
fig.add_trace(go.Scatter(
    x=top_purpose['purpose'],
    y=top_purpose['avg_int_rate'],
    
    name='<b>Avg Interest Rate</b>',
    mode='lines+markers',
    
    hovertemplate='%{y:.2%}',

    line=dict(color='deepskyblue', width=4, shape='spline'),
    marker=dict(size=8, color='#0A2540', line=dict(width=2, color='white')),
    fillcolor='rgba(30,136,229,0.15)',
    yaxis='y2'))
fig.update_layout(
    title=dict(text='<b>Loan Purpose vs Default Risk & Interest Rate</b>', x=0.5),
    xaxis=dict(
        title='<b>Loan Purpose</b>',tickangle=30,tickfont=dict(size=14)),
    yaxis=dict(
        title='<b>Default Rate (%)</b>',side='left',ticksuffix='%',tickfont=dict(size=14),showgrid=True),
    yaxis2=dict(
        title='<b>Avg Interest Rate</b>',
        overlaying='y',
        side='right',
        tickformat='.0%',
        tickfont=dict(size=14)),
    template='simple_white',
    legend=dict(
        x=0.8,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'),
    hovermode='x unified')
fig.show()

### Pricing & Repayment Behavior

    Interest rate (int_rate) distribution by grade — does pricing match risk?

In [33]:
avg_rate = df.groupby('grade')['int_rate'].mean().reset_index()
avg_rate['int_rate'] = avg_rate['int_rate'] * 100

avg_rate

,grade,int_rate
0,A,7.351971
1,B,11.029088
2,C,13.546922
3,D,15.710540
4,E,17.705230
5,F,19.744008
6,G,21.400639


In [34]:
fig = go.Figure(go.Bar(
    x=avg_rate['grade'],
    y=avg_rate['int_rate'],

    text=(avg_rate['int_rate'] * 100).round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    name='Loan Grade',
    marker=dict(color='rgb(26, 118, 255)'),

    hovertemplate=
    '<b>Grade:</b> %{x}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=avg_rate['grade'],
    y=avg_rate['int_rate'],

    mode='lines+markers',
    text=(avg_rate['int_rate'] * 100).round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='top center',

    line=dict(color='black', dash='dash'),
    name='Mean Interest Rate',

    hovertemplate=
    '<b>Avg Interest Rate:</b> %{y:.2%}<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>Average Interest Rate by Loan Grade</b>', x=0.5),
    xaxis_title='<b>Loan Grade</b>',
    yaxis=dict(title=dict(text='<b>Interest Rate</b>'),showgrid=True),
    template='simple_white',
    hovermode='x unified',
    bargap=0.45,
    legend=dict(
        x=0.8,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'))
fig.show()



The interest rate distribution shows a clear increasing trend from Grade A to Grade G, indicating that pricing is strongly aligned with borrower risk. Lower-risk grades receive lower interest rates, while higher-risk grades are charged significantly higher rates, reflecting proper risk-based pricing behavior.

    Installment amount vs default rate 

In [35]:
# Create default flag
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)

# Correlation
corr = df['installment'].corr(df['is_default'])
print(f"Correlation between Installment & Default: {corr:.3f}")

Correlation between Installment & Default: 0.027


In [36]:
bins = [0, 150, 250, 350, 500, df['installment'].max()]
labels = ['0–150', '150–250', '250–350', '350–500', '500+']

df['inst_bin'] = pd.cut(df['installment'], bins=bins, labels=labels)


In [37]:
inst_analysis = df.groupby('inst_bin').agg(default_rate=('is_default', lambda x: x.mean() * 100),
                                           avg_installment=('installment', 'mean'),
                                           total_loans=('id', 'count')).reset_index()
inst_analysis

,inst_bin,default_rate,avg_installment,total_loans
0,0–150,13.528703,97.268933,7473
1,150–250,12.627137,196.330714,9242
2,250–350,13.190777,300.934476,7763
3,350–500,14.512023,418.345612,7070
4,500+,15.722823,679.258497,7028


In [38]:
fig = go.Figure(go.Bar(
    x=inst_analysis['inst_bin'].astype(str),
    y=inst_analysis['default_rate'],

    text=inst_analysis['default_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    customdata=inst_analysis[['avg_installment', 'total_loans']],
    hovertemplate=
    '<b>Installment Range:</b> %{x}<br>' +
    '<b>Default Rate:</b> %{y:.2f}%<br>' +
    '<b>Avg Installment:</b> %{customdata[0]:,.0f}<br>' +
    '<b>Total Loans:</b> %{customdata[1]:,}<extra></extra>',

    marker=dict(
        color=inst_analysis['default_rate'],   
        colorscale='YlOrRd',                   
        colorbar=dict(title='<b>Default %</b>'))
))
fig.update_layout(
    title=dict(text='<b>Default Rate by Installment Level</b>', x=0.5),
    xaxis_title='<b>Installment Range</b>',
    yaxis=dict(title=dict(text='<b>Default Rate (%)</b>'),showgrid=True),
    template='simple_white',
    bargap=0.45)
fig.show()

    Loan term (36 vs 60 months) impact on default and total payment

In [39]:
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)

term_analysis = df.groupby('term').agg(default_rate=('is_default', lambda x: x.mean() * 100),
                                             total_payment=('total_payment', 'sum'),
                                             avg_payment=('total_payment', 'mean'),
                                             total_loans=('id', 'count')).reset_index()

term_analysis

,term,default_rate,total_payment,avg_payment,total_loans
0,36,10.705812,294709458,10436.996069,28237
1,60,22.342586,178361475,17251.327498,10339


In [40]:
# Scale total_payment (convert to millions for readability)
term_analysis['total_payment_m'] = term_analysis['total_payment'] / 1_000_000

fig = go.Figure()
fig.add_trace(go.Bar(
    x=term_analysis['term'],
    y=term_analysis['default_rate'],

    name='<b>Default Rate (%)</b>',
    text=term_analysis['default_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    marker=dict(color='#E53935'),

    hovertemplate=
    '<b>Term:</b> %{x} months<br>' +
    '<b>Default Rate:</b> %{y:.2f}% <extra></extra>'
))
fig.add_trace(go.Bar(
    x=term_analysis['term'],
    y=term_analysis['total_payment_m'],

    name='<b>Total Payment (Millions)</b>',
    text=term_analysis['total_payment_m'].round(1),
    texttemplate='<b>%{text}M</b>',
    textposition='outside',

    marker=dict(color='#1E88E5'),

    customdata=term_analysis[['total_loans']],
    hovertemplate=
    '<b>Total Loans:</b> %{customdata[0]:,}<br>' +
    '<b>Total Payment:</b> %{y:.1f} Million<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>Loan Term Comparison: Default Risk vs Total Payment</b>', x=0.5),
    xaxis_title='<b>Loan Term (Months)</b>',
    yaxis_title='<b>Values (Scaled)</b>',
    barmode='group',
    template='plotly_white',
    bargap=0.5,
    hovermode='x unified',
    legend=dict(
        x=0.8,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'))
fig.show()

    Income-to-loan ratio vs repayment success

In [41]:
# Calculate Income Ratio
df['income_loan_ratio'] = df['annual_income'] / df['loan_amount']

df['is_repaid'] = (df['loan_status'] == 'Fully Paid').astype(int)
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)

# Create Ratio Bins/Sections
df['ratio_bin'] = pd.qcut(df['income_loan_ratio'], 5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

In [42]:
ratio_analysis = df.groupby('ratio_bin').agg(
    repayment_rate=('is_repaid', lambda x: x.mean() * 100),
    default_rate=('is_default', lambda x: x.mean() * 100),
    avg_ratio=('income_loan_ratio', 'mean'),
    total_loans=('id', 'count')).reset_index()

ratio_analysis

,ratio_bin,repayment_rate,default_rate,avg_ratio,total_loans
0,Very Low,74.175474,20.221571,2.814460,7853
1,Low,81.875742,14.641868,4.357774,7581
2,Medium,85.310515,12.485414,6.137240,7713
3,High,87.388205,10.913804,9.169444,7715
4,Very High,88.034742,10.759658,22.128779,7714


In [43]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=ratio_analysis['ratio_bin'],
    y=ratio_analysis['repayment_rate'],

    name='<b>Repayment Rate (%)</b>',
    text=ratio_analysis['repayment_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    customdata=ratio_analysis[['avg_ratio', 'total_loans']],
    hovertemplate=
    '<b>Total Loans:</b> %{customdata[1]:,}</br>' +
    '<b>Ratio Level:</b> %{x}<br>' +
    '<b>Repayment:</b> %{y:.2f}%<br>' +
    '<b>Avg Ratio:</b> %{customdata[0]:.2f}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=ratio_analysis['ratio_bin'],
    y=ratio_analysis['default_rate'],

    name='<b>Default Rate (%)</b>',
    text=ratio_analysis['default_rate'].round(2),
    texttemplate='<b>%{text}%</b>',
    textposition='outside',

    hovertemplate='%{y:,.2f}%' 
))
fig.update_layout(
    title=dict(text='<b>Income-to-Loan Ratio vs Repayment Behavior</b>', x=0.5),
    xaxis=dict(
        title=dict(
            text='<b>Income-to-Loan Ratio Level</b>',
            font=dict(size=16))),
    yaxis=dict(
        title=dict(
            text='<b>Default Rate (%)</b>',
            font=dict(size=16))),
    barmode='group',
    template='plotly_white',
    bargap=0.2,
    hovermode='x unified',
    legend=dict(
        x=0.8,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)'))
fig.show()

### Geographic & Behavioral Insights

    Top 10 states (address_state) by loan volume vs default rate 

In [44]:
state_analysis = df.groupby('address_state').agg(
    total_loans=('id', 'count'),
    default_rate=('is_default', lambda x: x.mean() * 100),
    total_amount=('loan_amount', 'sum')).reset_index()

top_states = state_analysis.sort_values('total_loans', ascending=False).head(10)

In [62]:
top_states

,address_state,total_loans,default_rate,total_amount
4,CA,6894,15.332173,78484125
33,NY,3701,12.645231,42077050
9,FL,2773,17.273711,30046125
42,TX,2664,11.298799,31236650
30,NJ,1822,14.983535,21657475
14,IL,1486,12.920592,17124225
37,PA,1482,11.403509,15826525
44,VA,1375,12.363636,15982650
10,GA,1355,15.276753,15480325
19,MA,1310,11.450382,15051000


In [46]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=top_states['address_state'],
    y=top_states['total_loans'],

    name='<b>Total Loans</b>',
    text=top_states['total_loans'],
    texttemplate='<b>%{text:,.0f}</b>',
    textposition='outside',
    textfont=dict(size=12,color='black'),

    marker=dict(color=top_states['total_loans'], colorscale='ylorrd'),
    yaxis='y1',
    
    hovertemplate=
    '<b>Total Loans:</b> %{text:,.0f}'
))
fig.add_trace(go.Scatter(
    x=top_states['address_state'],
    y=top_states['default_rate'],

    name='<b>Default Rate</b>',
    mode='lines+markers',

    text=top_states['default_rate'].round(1),
    texttemplate='<b>%{text}%</b>',
    textposition='top center',

    line=dict(color='deepskyblue', width=4, shape='spline'),
    marker=dict(size=8, color='#0A2540', line=dict(width=2, color='white')),
    fill = 'tozeroy',
    fillcolor='rgba(30,136,229,0.15)',
    yaxis='y2',

    hovertemplate=
    '<b>Default Rate:</b> %{y:.2f}%<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>Top States: Loan Volume vs Default Risk</b>', x=0.5),
    xaxis=dict(
        title=dict(
            text='<b>State</b>',
            font=dict(size=18))),
    yaxis=dict(
         title=dict(
            text='<b>Total Loans</b>',
             font=dict(size=16)),
        side='left'),
    yaxis2=dict(
        title=dict(
            text='<b>Default Rate (%)</b>',
             font=dict(size=16)),
        overlaying='y',
        side='right',
        range=[0, 35]),
    template='simple_white',
    legend=dict(
        x=0.85,
        y=1.15,
        font=dict(size=12),
        bgcolor='rgb(255,255,255,0.8)',
        bordercolor='black',
        borderwidth=1),
    hovermode='x unified')
fig.show()

    Application_type (INDIVIDUAL vs others) performance

In [47]:
df['is_default'] = (df['loan_status'] == 'Charged Off').astype(int)
df['is_repaid'] = (df['loan_status'] == 'Fully Paid').astype(int)

In [48]:
app_analysis = df.groupby('application_type').agg(
    Total_loans=('id', 'count'),
    Default_rate=('is_default', lambda x: x.mean() * 100),
    Repayment_rate=('is_repaid', lambda x: x.mean() * 100),
    Avg_loan=('loan_amount', 'mean'),
    Avg_interest=('int_rate', lambda x: x.mean() * 100),
    Total_amount=('loan_amount', 'sum')).reset_index()

In [49]:
app_analysis['Total_loans'] = app_analysis['Total_loans'].apply(lambda x: f"{x:,}")
app_analysis['Avg_loan'] = app_analysis['Avg_loan'].apply(lambda x: f"{x:,.0f}")
app_analysis['Total_amount'] = app_analysis['Total_amount'].apply(lambda x: f"{x:,.0f}")

app_analysis['Default_rate'] = app_analysis['Default_rate'].apply(lambda x: f"{x:.2f}%")
app_analysis['Repayment_rate'] = app_analysis['Repayment_rate'].apply(lambda x: f"{x:.2f}%")
app_analysis['Avg_interest'] = app_analysis['Avg_interest'].apply(lambda x: f"{x:.2f}%")

In [50]:
app_analysis

,application_type,Total_loans,Default_rate,Repayment_rate,Avg_loan,Avg_interest,Total_amount
0,INDIVIDUAL,"38,576",13.82%,83.33%,"11,296",12.05%,"435,757,075"


### Revenue & Risk Signals

    Total expected revenue proxy = total_payment for Fully Paid loans

In [51]:
# Revenue only from Fully Paid loans
total_expected_revenue = df.loc[df['loan_status'] == 'Fully Paid', 'total_payment'].sum()

print(f"Revenue only from Fully Paid loans: {total_expected_revenue:,.0f}$")

Revenue only from Fully Paid loans: 411,586,256$


In [52]:
revenue_summary = df.groupby('loan_status').agg(total_loans=('id', 'count'),total_payment=('total_payment', 'sum')).reset_index()

revenue_summary

,loan_status,total_loans,total_payment
0,Charged Off,5333,37284763
1,Current,1098,24199914
2,Fully Paid,32145,411586256


In [53]:
rev_analysis = df.groupby('loan_status').agg(total_payment=('total_payment', 'sum'),total_loans=('id', 'count')).reset_index()

rev_analysis['payment_share_%'] = (rev_analysis['total_payment'] / rev_analysis['total_payment'].sum()) * 100

rev_analysis

,loan_status,total_payment,total_loans,payment_share_%
0,Charged Off,37284763,5333,7.881432
1,Current,24199914,1098,5.115494
2,Fully Paid,411586256,32145,87.003074


In [54]:
fig = go.Figure(go.Pie(
    labels=rev_analysis['loan_status'],
    values=rev_analysis['total_payment'],

    hole=0.55,
    pull=[0.05, 0.05, 0.05],

    textinfo='percent+label',
    textfont=dict(size=11, color='white'),

    marker=dict(colors=['#0A2540', '#1E88E5', '#43A0999','#8E24AA'],line=dict(color='white', width=2)),
    hovertemplate="<b>%{label}<br>Total Payment:</b> $%{value:,.0f}<extra></extra>",
))
fig.update_layout(
    title=dict(text='<b>Total Payment by Loan Status</b>', x=0.5, font=dict(size=22)),
    xaxis_title='<b>Loan Status</b>',
    yaxis_title='<b>Total Payment</b>',
    template='plotly_white',
    annotations=[dict(
        text="<b>Payment<br>Distribution</b>",
        x=0.5,
        y=0.5,
        font=dict(size=18),
        showarrow=False)],  
    showlegend=True,
    legend=dict(
        title = "<b>Payment Status</b>",
        orientation="v",
        x=0.75,
        y=1.05,
        font=dict(size=12)
    ))
fig.show()

    Estimated loss from Charged Off loans

In [55]:
df['loss_amount'] = np.where(
    df['loan_status'] == 'Charged Off',df['loan_amount'] - df['total_payment'],0)

total_loss = df['loss_amount'].sum()

print(f"Estimated Loss From Charged Off Loans are: ${total_loss:,.0f}")

Estimated Loss From Charged Off Loans are: $28,247,462


In [56]:
loss_summary = df.groupby('loan_status').agg(
    total_loans=('id', 'count'),
    total_loan_amount=('loan_amount', 'sum'),
    total_payment=('total_payment', 'sum'),
    total_loss=('loss_amount', 'sum')
).reset_index()

loss_summary

,loan_status,total_loans,total_loan_amount,total_payment,total_loss
0,Charged Off,5333,65532225,37284763,28247462
1,Current,1098,18866500,24199914,0
2,Fully Paid,32145,351358350,411586256,0


    Correlation heatmap between numerical features (annual_income, dti, int_rate, installment, loan_amount, total_payment)

In [57]:
num_cols = ['annual_income','dti','int_rate','installment','loan_amount','total_payment']
corr_matrix = df[num_cols].corr()

corr_matrix

,annual_income,dti,int_rate,installment,loan_amount,total_payment
annual_income,1.000000,-0.124877,0.050665,0.267362,0.268183,0.254611
dti,-0.124877,1.000000,0.112978,0.053381,0.065713,0.064319
int_rate,0.050665,0.112978,1.000000,0.280959,0.308243,0.308844
installment,0.267362,0.053381,0.280959,1.000000,0.929655,0.857745
loan_amount,0.268183,0.065713,0.308243,0.929655,1.000000,0.887490
total_payment,0.254611,0.064319,0.308844,0.857745,0.887490,1.000000


In [58]:
fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,             
    x=corr_matrix.columns,            
    y=corr_matrix.index,                 

    colorscale='Blues',
    text=corr_matrix.round(2),
    texttemplate='%{text}',
    colorbar=dict(title='<b>Correlation</b>'),

    hovertemplate =
        '<b>%{x} vs %{y}</b><br>' +
        'Correlation: %{z:.2f}<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>Correlation Heatmap: Financial Features</b>', x=0.5),
    xaxis_title='<b>Features</b>',
    yaxis_title='<b>Features</b>',
    template='simple_white')

fig.show()

## Business Recommendations(Suggestions)

1. Reduce exposure to high-risk loan grades (E–G) as they have significantly higher default rates. Focus more on safer grades like A and B.
2. Limit long-term loans (60 months) since they show much higher default rates compared to 36-month loans.
3. Strengthen credit policies for high-risk purposes such as small business loans, which have the highest default rates.
4. Encourage lending to higher-income customers, as default risk decreases significantly with increasing income levels.
5. Apply stricter screening for borrowers with high debt-to-income (DTI) ratios, especially in the 15–25% range where risk peaks.
6. Promote loans with higher income-to-loan ratios, as they show better repayment performance and lower default rates.
7. Improve monitoring of high installment loans, as default risk increases with higher installment amounts.
8. Maintain risk-based pricing strategy, as interest rates are effectively aligned with loan risk across grades.
9. Focus marketing and lending in lower default regions (e.g., Texas, Pennsylvania) and review strategies in high-risk states like Florida.
10. Minimize losses from charged-off loans by improving early risk detection and strengthening collection strategies.